# Mac-native SC2 Transformer

This notebook is the supported macOS path: decode public `.SC2Replay` events with `s2protocol`, train a modern PyTorch Transformer, then load a checkpoint into a small PySC2 agent. It intentionally does not use AlphaStar Unplugged's Linux-only C++ converter.

In [1]:
from pathlib import Path
import os, subprocess, sys, torch

REPO_ROOT = Path.cwd().resolve()
if not (REPO_ROOT / 'mac_sc2').exists():
    REPO_ROOT = REPO_ROOT.parent
PYTHON = REPO_ROOT / '.conda-alphastar' / 'bin' / 'python'
SC2PATH = Path('/Applications/StarCraft II')
assert PYTHON.exists(), 'Select the AlphaStar Mac (PyTorch + SC2) kernel.'
assert SC2PATH.exists(), 'Install StarCraft II first.'
print('Python:', PYTHON)
print('PyTorch:', torch.__version__, '| MPS:', torch.backends.mps.is_available())
print('SC2:', SC2PATH)

Python: /Users/johnnylee/PycharmProjects/alphastar/.conda-alphastar/bin/python
PyTorch: 2.8.0 | MPS: True
SC2: /Applications/StarCraft II


In [2]:
# Train behavior cloning on actual public replay command events.
command = [str(PYTHON), 'mac_sc2/train_replay_transformer.py', '--epochs', '4', '--max-replays', '3']
result = subprocess.run(command, cwd=REPO_ROOT, text=True, capture_output=True)
print(result.stdout)
if result.returncode:
    print(result.stderr)
    raise RuntimeError(f'replay Transformer failed with code {result.returncode}')

Replays=3, player sequences=6, examples=1995, vocab=67, device=mps
epoch 1: loss=3.4601, accuracy=0.202
epoch 2: loss=2.7458, accuracy=0.299
epoch 3: loss=2.5014, accuracy=0.321
epoch 4: loss=2.3657, accuracy=0.343
Saved checkpoint: /Users/johnnylee/PycharmProjects/alphastar/mac_sc2/artifacts/replay_transformer.pt



## Live SC2 adapter

The replay Transformer above learns historical ability-token distributions. A raw AlphaStar ability token cannot be safely issued to the current SC2 client because replay build 74741 and installed build 97563 have different action schemas. The following deliberately narrow `MoveToBeacon` controller verifies the complete deployable loop: Transformer checkpoint → live PyTorch inference → legal PySC2 action.

In [3]:
# Train the small coordinate-policy Transformer used by the live smoke test.
command = [str(PYTHON), 'mac_sc2/train_beacon_transformer.py', '--epochs', '30']
result = subprocess.run(command, cwd=REPO_ROOT, text=True, capture_output=True)
print(result.stdout)
if result.returncode:
    print(result.stderr)
    raise RuntimeError(f'live policy training failed with code {result.returncode}')

epoch 1: accuracy=0.000
epoch 5: accuracy=0.035
epoch 10: accuracy=0.176
epoch 15: accuracy=0.516
epoch 20: accuracy=0.879
epoch 25: accuracy=0.980
epoch 30: accuracy=0.996
Saved checkpoint: /Users/johnnylee/PycharmProjects/alphastar/mac_sc2/artifacts/beacon_transformer.pt



In [4]:
# Launch SC2 headlessly, load the checkpoint, and execute 64 model decisions.
environment = os.environ | {'SC2PATH': str(SC2PATH)}
command = [str(PYTHON), 'mac_sc2/play_beacon_transformer.py', '--steps=64', '--checkpoint=mac_sc2/artifacts/beacon_transformer.pt']
result = subprocess.run(command, cwd=REPO_ROOT, text=True, capture_output=True, env=environment)
print(result.stdout)
if result.returncode:
    print(result.stderr)
    raise RuntimeError(f'SC2 agent failed with code {result.returncode}')

pygame 2.6.1 (SDL 2.28.4, Python 3.9.25)
Hello from the pygame community. https://www.pygame.org/contribute.html
Finished SC2 inference run after 64 agent steps.

